# PHASE — ESC-50 dry run

Closes the Phase 3 exit criterion: the supervised baseline must train end to end and reach a sane accuracy.

This is a **plumbing test, not a result**. It exists so that when DeepShip training misbehaves, the pipeline is not a suspect.

Expected range for ResNet-18 on ESC-50 log-Mel: **roughly 60-80% accuracy**. Anything far below that points at the trainer and must be resolved before Phase 6.

Runtime -> Change runtime type -> **T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Repository and dependencies

In [ ]:
import os, pathlib

REPO = pathlib.Path('/content/deepsonar_v2')
if not REPO.exists():
    !git clone --depth 1 https://github.com/Adhi-1004/deepsonar_v2.git {REPO}
os.chdir(REPO)
print(pathlib.Path.cwd())

In [ ]:
%pip install -q -e '.[data]' 2>&1 | tail -3

import importlib.util
import torch, phase

print('phase', phase.__version__)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print()
for module in ('kaggle', 'librosa', 'soundfile', 'wandb', 'sklearn', 'huggingface_hub'):
    found = importlib.util.find_spec(module) is not None
    print(f"{module:16s} {'installed' if found else 'MISSING'}")


## 2. ESC-50

Needs a Kaggle token. Upload `kaggle.json` when prompted, or set `KAGGLE_USERNAME` and `KAGGLE_KEY`.

In [ ]:
import pathlib

cred = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
if not cred.exists() and not os.environ.get('KAGGLE_KEY'):
    from google.colab import files
    cred.parent.mkdir(parents=True, exist_ok=True)
    up = files.upload()
    cred.write_bytes(next(iter(up.values())))
    cred.chmod(0o600)
print('credentials ready')

In [ ]:
!python -m phase.data.download --dataset esc50
!python -m phase.data.download --check --all

## 3. Manifest, leakage-safe splits, feature cache

Splits are grouped by source clip, so the several takes of one Freesound recording never straddle a split.

In [ ]:
!python scripts/02_build_manifests.py --dataset esc50 2>&1 | head -12
!python scripts/03_build_splits.py --dataset esc50 2>&1 | grep -E '==>|train |val |test |wrote|SKIPPED' | head -20

In [ ]:
!python scripts/04_segment_and_cache.py --dataset esc50 --features logmel 2>&1 | grep -vE 'per class|per split|cached'

## 4. Confirm the split is leakage-free before training

If this prints anything other than zero, stop.

In [ ]:
import json

index = json.load(open('data/cache/esc50_fold_0_windows.json'))
groups = {}
for w in index['windows']:
    groups.setdefault(w['recording_id'], set()).add(w['split'])
straddling = sum(1 for v in groups.values() if len(v) > 1)
print(f"windows {index['n_windows']}  recordings {len(groups)}  straddling splits {straddling}")
assert straddling == 0

## 5. Train — three seeds

`configs/eval/finetune.yaml` drives every hyperparameter. Nothing is hardcoded here.

In [ ]:
!python scripts/06_evaluate.py \
    --config configs/eval/finetune.yaml \
    --cache data/cache/esc50_fold_0_logmel \
    --seeds 0 1 2 \
    --wandb-mode offline 2>&1 | grep -vE '^wandb|artifact'

## 6. Verdict

In [ ]:
import glob, json

path = sorted(glob.glob('results/tables/*seeds.json'))[-1]
summary = json.load(open(path))['summary']

print(path, '\n')
for key, stats in summary.items():
    print(f"{key:12s} {stats['mean']:.4f} +/- {stats['std']:.4f}  (n={stats['n']})")

acc = summary['accuracy']['mean']
print()
if acc >= 0.55:
    print(f'PASS - {acc:.1%} is a sane ESC-50 accuracy. Phase 3 exit criterion met.')
elif acc >= 0.20:
    print(f'MARGINAL - {acc:.1%}. Well above the 2% chance level, so the pipeline works,')
    print('but below the expected 60-80%. Try more epochs before suspecting the trainer.')
else:
    print(f'FAIL - {acc:.1%} is close to the 2% chance level. The trainer has a bug;')
    print('resolve it before Phase 6 rather than during pretraining.')

## 7. Download the result

Send me this file and I will record it in `docs/results_log.md`.

In [ ]:
from google.colab import files
files.download(path)

## 8. Augmentation cost on GPU

Phase 4 measured the physics pipeline at **~100-170 ms per view-window on CPU**, which at batch 128 would be roughly 40 s per batch — not runnable for 200 epochs.

These are elementwise complex ops on a few million elements, exactly what a GPU is for, but the speedup has **not been measured**. This cell measures it. Phase 6 should not start until the number here is acceptable.

In [ ]:
import time, torch
from phase.config import AugmentConfig, load_config
from phase.augment import Pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = load_config('configs/augment/physics.yaml', AugmentConfig)
pipeline = Pipeline(config)
generator = torch.Generator(device='cpu').manual_seed(0)

for batch_size in (8, 32, 128):
    x = torch.randn(batch_size, 32000 * 30, device=device)
    pipeline.views(x, 32000, generator)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(3):
        pipeline.views(x, 32000, generator)
    if device == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.time() - start) / 3
    print(f'B={batch_size:3d}  {elapsed:6.2f} s/batch  {1000*elapsed/(2*batch_size):6.1f} ms per view-window')

print()
print('CPU reference: ~100-170 ms per view-window')
print('At 50 batches/epoch and 200 epochs, keep s/batch well under ~2 s or Phase 6 will not finish.')